# v28 Standard — YNU CV Rerank + Overfit/Underfit Diagnostics

Purpose:
- Start from v27 standard if available; otherwise v25.
- MC is true-frozen.
- Tune only YNU.
- Print explicit overfit/underfit diagnostics.
- Use CV/holdout to avoid trusting full-val grid too much.

In [ ]:
import os, re, json, glob, random, math, statistics
from pathlib import Path
from collections import Counter, defaultdict

LABELS = ["A","B","C","D","Yes","No","Unknown"]
MC_LABELS = {"A","B","C","D"}
YNU_LABELS = {"Yes","No","Unknown"}
OUT_DIR = Path("/kaggle/working/v28_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

def norm_answer(x):
    if x is None: return "UNPARSEABLE"
    s = str(x).strip()
    if not s: return "UNPARSEABLE"
    low = s.lower()
    if low in ["a","option a","answer a"]: return "A"
    if low in ["b","option b","answer b"]: return "B"
    if low in ["c","option c","answer c"]: return "C"
    if low in ["d","option d","answer d"]: return "D"
    if low in ["yes","true","entailed","supported"]: return "Yes"
    if low in ["no","false","not entailed","unsupported","contradicted"]: return "No"
    if low in ["unknown","uncertain","not enough information","cannot determine"]: return "Unknown"
    m = re.search(r"\b(final answer|answer)\s*[:\-]\s*(A|B|C|D|Yes|No|Unknown)\b", s, flags=re.I)
    if m: return norm_answer(m.group(2))
    m = re.search(r"\b(A|B|C|D|Yes|No|Unknown)\b", s, flags=re.I)
    if m: return norm_answer(m.group(1))
    return "UNPARSEABLE"

def to_jsonable(x):
    try:
        import numpy as np
        if isinstance(x, (np.integer,)): return int(x)
        if isinstance(x, (np.floating,)): return float(x)
        if isinstance(x, (np.bool_,)): return bool(x)
    except Exception:
        pass
    if isinstance(x, dict): return {str(k): to_jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [to_jsonable(v) for v in x]
    return x

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(obj), f, ensure_ascii=False, indent=2)
    print("Saved:", path)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def resolve_one(patterns, required=True, label="file"):
    hits = []
    for pat in patterns:
        hits += glob.glob(pat, recursive=True)
    hits = sorted(set(hits))
    print(f"{label} candidates:", hits[:20], "..." if len(hits) > 20 else "")
    if not hits:
        if required:
            raise FileNotFoundError(f"Missing {label}; checked {patterns}")
        return None
    # Prefer input dataset, explicit new version, and shorter paths
    hits = sorted(hits, key=lambda p: (
        0 if "/kaggle/input/" in p else 1,
        0 if "v27_standard_preds" in Path(p).name else 1,
        1 if "ORACLE" in Path(p).name.upper() else 0,
        len(p)
    ))
    print("Using", label, ":", hits[0])
    return Path(hits[0])

def compute_metrics(rows, name="metric"):
    y_true = [norm_answer(r.get("gold")) for r in rows]
    y_pred = [norm_answer(r.get("pred")) for r in rows]
    n = len(rows)
    acc = sum(t == p for t,p in zip(y_true,y_pred)) / max(1,n)
    per, f1s, weights = {}, [], []
    for lab in LABELS:
        tp = sum(t == lab and p == lab for t,p in zip(y_true,y_pred))
        fp = sum(t != lab and p == lab for t,p in zip(y_true,y_pred))
        fn = sum(t == lab and p != lab for t,p in zip(y_true,y_pred))
        support = sum(t == lab for t in y_true)
        pred_count = sum(p == lab for p in y_pred)
        precision = tp/(tp+fp) if tp+fp else 0.0
        recall = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
        per[lab] = dict(precision=precision, recall=recall, f1=f1, support=support, pred_count=pred_count, tp=tp, fp=fp, fn=fn)
        f1s.append(f1); weights.append(support)
    return dict(
        name=name, n=n, acc=acc, macro_f1=sum(f1s)/len(f1s),
        weighted_f1=sum(f*w for f,w in zip(f1s,weights))/max(1,sum(weights)),
        mc_macro_f1=sum(per[x]["f1"] for x in ["A","B","C","D"])/4,
        ynu_macro_f1=sum(per[x]["f1"] for x in ["Yes","No","Unknown"])/3,
        per_label=per,
    )

def print_metric(m):
    print("="*100)
    print(m["name"])
    print("="*100)
    print(f"N={m['n']} acc={m['acc']:.4f} macro={m['macro_f1']:.4f} weighted={m['weighted_f1']:.4f}")
    print(f"MC={m['mc_macro_f1']:.4f} YNU={m['ynu_macro_f1']:.4f}")
    print("-"*100)
    print(f"{'Label':<10} {'P':>8} {'R':>8} {'F1':>8} {'Gold':>7} {'Pred':>7} {'Flag':>12}")
    for lab in LABELS:
        d = m["per_label"][lab]
        flag = "OK"
        if d["support"] and d["f1"] < 0.35: flag = "LOW_F1"
        if d["support"] and d["recall"] < 0.25: flag = "LOW_RECALL"
        if d["pred_count"] == 0 and d["support"] > 0: flag = "COLLAPSE"
        print(f"{lab:<10} {d['precision']:8.3f} {d['recall']:8.3f} {d['f1']:8.3f} {d['support']:7d} {d['pred_count']:7d} {flag:>12}")

def load_base_preds():
    p = resolve_one([
        "/kaggle/input/**/v27_standard_preds.json",
        "/kaggle/working/**/v27_standard_preds.json",
        "/kaggle/input/**/v25_operational_preds*.json",
        "/kaggle/working/**/v25_operational_preds*.json",
    ], required=True, label="base preds v27/v25")
    rows = load_json(p)
    print("Loaded base rows:", len(rows))
    return rows, p

def load_candidates(required=True):
    p = resolve_one([
        "/kaggle/input/**/v23_val_candidates.json",
        "/kaggle/input/**/v23*candidates*.json",
        "/kaggle/working/**/v23_val_candidates.json",
        "/kaggle/working/**/v23*candidates*.json",
    ], required=required, label="v23 candidates")
    if p is None: return None, None
    rows = load_json(p)
    print("Loaded candidate rows:", len(rows))
    return rows, p

def assert_mc_frozen(base, rows):
    bad = []
    for b,r in zip(base, rows):
        if bool(b.get("is_mc")) and norm_answer(b.get("pred")) != norm_answer(r.get("pred")):
            bad.append((b.get("idx"), b.get("pred"), r.get("pred")))
    if bad:
        print("MC freeze violations:", bad[:20])
        raise AssertionError(f"MC freeze violated in {len(bad)} rows")
    print("MC freeze check: PASS")

def cand_map(candidates):
    out = {}
    for i,r in enumerate(candidates or []):
        idx = r.get("idx", r.get("id", r.get("record_idx", i)))
        out[int(idx)] = r
    return out

def c_answer(c):
    return norm_answer(c.get("answer") or c.get("pred") or c.get("final_answer"))

def get_cands(row, cmap):
    return (cmap.get(int(row.get("idx", -1))) or {}).get("candidates", [])

def cand_text(c):
    return str(c.get("text") or c.get("raw_text") or c.get("explanation") or "")

def explicit_final_answer(c):
    txt = cand_text(c)
    m = re.search(r"\bFinal Answer\s*:\s*(Yes|No|Unknown|A|B|C|D)\b", txt, flags=re.I)
    return norm_answer(m.group(1)) if m else None

def score_cand(c, p):
    ans = c_answer(c)
    final = explicit_final_answer(c)
    supp = c.get("supporting_premises") or c.get("supporting") or []
    try: supp_len = len(supp)
    except Exception: supp_len = 0
    txt = cand_text(c)
    score = 0.0
    score += p.get("w_vote", 1.0) * float(c.get("vote_share", 0.0) or 0.0)
    score += p.get("w_cite", 0.0) * (1.0 if supp_len > 0 else 0.0)
    score += p.get("w_short_supp", 0.0) * (1.0 if 1 <= supp_len <= 5 else 0.0)
    score += p.get("w_final_match", 0.0) * (1.0 if final == ans and final in YNU_LABELS else 0.0)
    if supp_len > 8: score -= p.get("p_long_supp", 0.0)
    if ans == "Yes": score += p.get("yes_boost", 0.0)
    if ans == "No": score -= p.get("no_penalty", 0.0)
    if ans == "Unknown": score -= p.get("unk_penalty", 0.0)
    if re.search(r"\b(however|contradict|cannot conclude|not enough|unknown)\b", txt, flags=re.I):
        score -= p.get("p_contra_text", 0.0)
    return score

def choose_ynu(row, cmap, params):
    cs = [c for c in get_cands(row, cmap) if c_answer(c) in YNU_LABELS]
    if not cs: return norm_answer(row.get("pred"))
    return c_answer(max(cs, key=lambda c: score_cand(c, params)))

def build_rows(base, cmap=None, params=None, mode="baseline"):
    rows = []
    for r in base:
        rr = dict(r)
        if bool(r.get("is_mc")):
            rr["pred"] = norm_answer(r.get("pred"))
            rr["source"] = "mc_frozen_from_base"
        else:
            if mode == "candidate_rerank" and cmap is not None:
                rr["pred"] = choose_ynu(r, cmap, params or {})
                rr["source"] = "ynu_candidate_rerank_v28"
            elif mode == "oracle_ynu" and cmap is not None:
                gold = norm_answer(r.get("gold"))
                answers = [c_answer(c) for c in get_cands(r, cmap)]
                rr["pred"] = gold if gold in answers else norm_answer(r.get("pred"))
                rr["source"] = "ynu_oracle_analysis"
            else:
                rr["pred"] = norm_answer(r.get("pred"))
                rr["source"] = "ynu_frozen_from_base"
        rows.append(rr)
    assert_mc_frozen(base, rows)
    return rows

def subset_rows(rows, idxs):
    s = set(idxs)
    return [r for i,r in enumerate(rows) if i in s]

def objective(m):
    return (
        m["macro_f1"]
        + 0.25*m["ynu_macro_f1"]
        + 0.12*m["per_label"]["Yes"]["f1"]
        + 0.12*m["per_label"]["Unknown"]["f1"]
        + 0.05*m["per_label"]["No"]["f1"]
    )

def param_grid():
    for w_vote in [0.0, 0.25, 0.5, 1.0]:
      for w_cite in [0.0, 0.2, 0.4]:
        for w_short in [0.0, 0.1, 0.2]:
          for w_final in [0.0, 0.2, 0.5]:
            for yes_boost in [0.0, 0.2, 0.4, 0.6]:
              for no_penalty in [0.0, 0.15, 0.3, 0.45]:
                for unk_penalty in [0.0, 0.15, 0.3, 0.45]:
                  yield dict(
                    w_vote=w_vote, w_cite=w_cite, w_short_supp=w_short, w_final_match=w_final,
                    yes_boost=yes_boost, no_penalty=no_penalty, unk_penalty=unk_penalty,
                    p_long_supp=0.2, p_contra_text=0.4
                  )

def evaluate_params(base_rows, cmap, params, idxs=None, name="eval"):
    rows = build_rows(base_rows, cmap, params, mode="candidate_rerank")
    if idxs is not None:
        rows_eval = subset_rows(rows, idxs)
    else:
        rows_eval = rows
    return compute_metrics(rows_eval, name)

def detect_fit_status(train_m, hold_m, oracle_m=None, base_m=None):
    train = train_m["macro_f1"]
    hold = hold_m["macro_f1"]
    gap = train - hold
    status = []
    if gap > 0.08:
        status.append("OVERFIT_RISK_HIGH")
    elif gap > 0.04:
        status.append("OVERFIT_RISK_MEDIUM")
    else:
        status.append("NO_STRONG_OVERFIT_SIGNAL")
    if train < 0.50 and hold < 0.50:
        status.append("UNDERFIT_RISK_HIGH_BOTH_LOW")
    elif oracle_m and (oracle_m["macro_f1"] - hold > 0.08):
        status.append("UNDERFIT_OR_SELECTION_GAP")
    else:
        status.append("NO_STRONG_UNDERFIT_SIGNAL")
    if base_m and hold < base_m["macro_f1"] - 0.01:
        status.append("WORSE_THAN_BASELINE")
    return status

def print_fit_report(report):
    print("="*100)
    print("OVERFIT / UNDERFIT DIAGNOSTIC")
    print("="*100)
    for k,v in report.items():
        print(f"{k}: {v}")

In [ ]:
# ==================================================================
# STAGE 1 -- Load base + candidates
# ==================================================================
base_rows, base_path = load_base_preds()
base_metric = compute_metrics(base_rows, "base_loaded_v27_or_v25")
print_metric(base_metric)

candidates, cand_path = load_candidates(required=True)
cmap = cand_map(candidates)
print("Candidate map size:", len(cmap))

In [ ]:
# ==================================================================
# STAGE 2 -- Oracle ceiling for underfit/selection-gap diagnosis
# ==================================================================
oracle_rows = build_rows(base_rows, cmap, {}, mode="oracle_ynu")
oracle_metric = compute_metrics(oracle_rows, "oracle_ynu_mc_frozen_ANALYSIS_ONLY")
print_metric(oracle_metric)

oracle_gap = oracle_metric["macro_f1"] - base_metric["macro_f1"]
print("Oracle macro gap:", oracle_gap)
if oracle_gap > 0.08:
    print("DIAG: LARGE_SELECTION_HEADROOM -> current reranker under-selects candidates / underfits selection.")
else:
    print("DIAG: SMALL_SELECTION_HEADROOM -> candidate set may be the bottleneck.")

In [ ]:
# ==================================================================
# STAGE 3 -- Train/holdout split for overfit/underfit detection
# ==================================================================
indices = list(range(len(base_rows)))
random.Random(SEED).shuffle(indices)
split = int(0.70 * len(indices))
train_idx = indices[:split]
hold_idx = indices[split:]

print("Train size:", len(train_idx), "Holdout size:", len(hold_idx))
print("Train gold dist:", Counter(norm_answer(base_rows[i].get("gold")) for i in train_idx))
print("Holdout gold dist:", Counter(norm_answer(base_rows[i].get("gold")) for i in hold_idx))

grid = []
for params in param_grid():
    m_train = evaluate_params(base_rows, cmap, params, idxs=train_idx, name="train")
    grid.append(dict(obj=objective(m_train), train_macro=m_train["macro_f1"], train_ynu=m_train["ynu_macro_f1"], **params))

grid = sorted(grid, key=lambda x: x["obj"], reverse=True)
best_params = {k:grid[0][k] for k in ["w_vote","w_cite","w_short_supp","w_final_match","yes_boost","no_penalty","unk_penalty","p_long_supp","p_contra_text"]}
print("Best params from TRAIN only:", best_params)

best_rows = build_rows(base_rows, cmap, best_params, mode="candidate_rerank")
train_metric = compute_metrics(subset_rows(best_rows, train_idx), "v28_train_metric")
hold_metric = compute_metrics(subset_rows(best_rows, hold_idx), "v28_holdout_metric")
full_attempt_metric = compute_metrics(best_rows, "v28_full_attempt_from_train_params")

print_metric(train_metric)
print_metric(hold_metric)
print_metric(full_attempt_metric)

fit_report = {
    "train_macro_f1": train_metric["macro_f1"],
    "holdout_macro_f1": hold_metric["macro_f1"],
    "train_minus_holdout_gap": train_metric["macro_f1"] - hold_metric["macro_f1"],
    "base_full_macro_f1": base_metric["macro_f1"],
    "attempt_full_macro_f1": full_attempt_metric["macro_f1"],
    "oracle_full_macro_f1": oracle_metric["macro_f1"],
    "oracle_minus_attempt_gap": oracle_metric["macro_f1"] - full_attempt_metric["macro_f1"],
    "status": detect_fit_status(train_metric, hold_metric, oracle_metric, base_metric),
}
print_fit_report(fit_report)

In [ ]:
# ==================================================================
# STAGE 4 -- 5-fold CV stability check
# ==================================================================
folds = [[] for _ in range(5)]
for j,i in enumerate(indices):
    folds[j % 5].append(i)

cv_rows = []
for f in range(5):
    val_idx = folds[f]
    tr_idx = [i for k,fold in enumerate(folds) if k != f for i in fold]
    local = []
    for params in param_grid():
        m_tr = evaluate_params(base_rows, cmap, params, idxs=tr_idx, name="cv_train")
        local.append(dict(obj=objective(m_tr), **params))
    local = sorted(local, key=lambda x: x["obj"], reverse=True)
    p = {k:local[0][k] for k in ["w_vote","w_cite","w_short_supp","w_final_match","yes_boost","no_penalty","unk_penalty","p_long_supp","p_contra_text"]}
    rows = build_rows(base_rows, cmap, p, mode="candidate_rerank")
    m_tr = compute_metrics(subset_rows(rows, tr_idx), f"fold{f}_train")
    m_val = compute_metrics(subset_rows(rows, val_idx), f"fold{f}_val")
    cv_rows.append(dict(fold=f, params=p, train_macro=m_tr["macro_f1"], val_macro=m_val["macro_f1"], gap=m_tr["macro_f1"]-m_val["macro_f1"],
                        train_ynu=m_tr["ynu_macro_f1"], val_ynu=m_val["ynu_macro_f1"]))

print("CV rows:")
for r in cv_rows:
    print(r)

val_macros = [r["val_macro"] for r in cv_rows]
gap_mean = sum(r["gap"] for r in cv_rows)/len(cv_rows)
cv_report = {
    "cv_val_macro_mean": sum(val_macros)/len(val_macros),
    "cv_val_macro_std": statistics.pstdev(val_macros),
    "cv_train_minus_val_gap_mean": gap_mean,
    "overfit_flag": "OVERFIT_RISK" if gap_mean > 0.05 else "NO_STRONG_OVERFIT_SIGNAL",
    "unstable_flag": "UNSTABLE_PARAMS_OR_SPLITS" if statistics.pstdev(val_macros) > 0.05 else "STABLE_ENOUGH",
}
print_fit_report(cv_report)

In [ ]:
# ==================================================================
# STAGE 5 -- Select output with rollback
# ==================================================================
# Guard: use v28 attempt only if full macro beats base and holdout is not much worse than base.
if (full_attempt_metric["macro_f1"] > base_metric["macro_f1"] + 1e-12) and (hold_metric["macro_f1"] >= base_metric["macro_f1"] - 0.04):
    selected_rows = best_rows
    selected_source = "v28_train_holdout_selected"
else:
    selected_rows = build_rows(base_rows, None, None, mode="baseline")
    selected_source = "rollback_to_base_due_to_fit_diagnostic"

selected_metric = compute_metrics(selected_rows, "v28_standard_SELECTED")
print("Selected source:", selected_source)
print_metric(selected_metric)

summary = dict(
    version="v28_standard_cv_fit_diagnostic",
    base_path=str(base_path),
    candidate_path=str(cand_path),
    base_metric=base_metric,
    oracle_metric=oracle_metric,
    best_params=best_params,
    train_metric=train_metric,
    holdout_metric=hold_metric,
    full_attempt_metric=full_attempt_metric,
    fit_report=fit_report,
    cv_rows=cv_rows,
    cv_report=cv_report,
    selected_source=selected_source,
    selected_metric=selected_metric,
)
save_json(summary, OUT_DIR / "v28_standard_summary.json")
save_json(selected_rows, OUT_DIR / "v28_standard_preds.json")
save_json(grid[:200], OUT_DIR / "v28_standard_grid_rows.json")